# AI Factory scheduling from a notebook

Submit **your own code** to AI Factory's scheduler from Python, and let it find room for it:

1. connect with your Rancher API key;
2. see your AI projects and their compute pools (what is free, which scheduler and runtimes each runs);
3. run your own Python program on two pods, and your own Ray program on a Ray cluster;
4. follow them: where they were placed, their logs and their result.

Install the SDK (`pip install ./sdk/python`) and set, before starting Jupyter:

```sh
export RANCHER_URL=https://rancher.example.com
export RANCHER_TOKEN=token-xxxxx:yyyy      # Rancher → Account & API Keys
export RANCHER_INSECURE=1                  # only for a self-signed lab certificate
```

In [ ]:
from rancher_ai import Client

PROJECT = "vision"            # your AI project
ai = Client(project=PROJECT)
ai.whoami()

## Where can my work run?

An AI project spans clusters; each cluster's nodes are compute pools (GPU or CPU). AI Factory's queue
places a run on the project's pool that fits it, and holds it until one does.

In [ ]:
ai.projects.list()

In [ ]:
ai.pools.list(PROJECT)

In [ ]:
ai.profiles.list("training")

## 1. My own Python program, on two pods

Plain Python: each pod reads its rank, the world size and rank 0's address from the environment
(`JOB_COMPLETION_INDEX`, `NNODES`, `RDZV_ENDPOINT`). Printing one `AIF_RESULT` line makes AI Factory
keep the result on the run.

In [ ]:
my_training = """
import json, os, random
rank, world = int(os.environ["JOB_COMPLETION_INDEX"]), int(os.environ["NNODES"])
random.seed(rank)
data = [(x, 2 * x + 1 + random.gauss(0, 0.1)) for x in (random.uniform(-1, 1) for _ in range(500))]
w = b = 0.0
for epoch in range(200):
    gw = sum((w * x + b - y) * x for x, y in data) * 2 / len(data)
    gb = sum((w * x + b - y) for x, y in data) * 2 / len(data)
    w, b = w - 0.1 * gw, b - 0.1 * gb
print(f"rank {rank}/{world}: w={w:.3f} b={b:.3f}", flush=True)
if rank == 0:
    ok = abs(w - 2) < 0.05 and abs(b - 1) < 0.05
    print("AIF_RESULT " + json.dumps({"test": "My training", "status": "pass" if ok else "fail",
          "checks": [{"name": "Learned y = 2x + 1", "ok": ok, "detail": f"w={w:.3f} b={b:.3f}"}]}))
"""

run = ai.runs.create("python-cpu-dev", code=my_training, workers=2)   # pool="..." to choose one yourself
run

In [ ]:
run.wait()          # Completed or Failed
run.status()        # where it was placed, and how it went

In [ ]:
run.logs(tail=10)

In [ ]:
run.result()

## 2. My own Ray program, on a Ray cluster

With the KubeRay add-on on the project's clusters, a Ray profile starts a Ray head and workers, runs
your program as the Ray driver, and removes the cluster when it ends. The driver can install what
it needs itself: `ray.init(runtime_env={"pip": ["torch"]})`.

In [ ]:
my_ray_program = """
import json, random, socket
import ray
ray.init()

@ray.remote
def sample(n):
    inside = sum(1 for _ in range(n) if random.random() ** 2 + random.random() ** 2 <= 1)
    return inside, socket.gethostname()

results = ray.get([sample.remote(100_000) for _ in range(200)])
pi = 4 * sum(i for i, _ in results) / (200 * 100_000)
hosts = sorted({h for _, h in results})
print(f"pi ~ {pi:.4f} from {len(hosts)} Ray workers", flush=True)
print("AIF_RESULT " + json.dumps({"test": "My Ray program", "status": "pass" if abs(pi - 3.1416) < 0.01 else "fail",
      "checks": [{"name": "Estimated pi", "ok": abs(pi - 3.1416) < 0.01, "detail": f"{pi:.4f}"}]}))
"""

ray_run = ai.runs.create("ray-cpu-dev", code=my_ray_program, workers=2)
ray_run.wait(), ray_run.result()

In [ ]:
ray_run.logs(tail=8)

## Clean up

Deleting a run uninstalls it from the cluster it ran on. Runs from profiles with a reclaim policy
are also requeued automatically if they sit idle while someone else waits for the pool.

In [ ]:
run.delete(); ray_run.delete()
ai.runs.list()